In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time

driver = webdriver.Chrome()
driver.maximize_window()
wait = WebDriverWait(driver, 10)

In [ ]:
try:
    driver.get("http://localhost:5173/")
    driver.execute_script("window.localStorage.clear(); window.sessionStorage.clear();")
    driver.get("http://localhost:5173/login")

    wait.until(EC.presence_of_element_located((By.ID, "username")))
    driver.find_element(By.ID, "username").send_keys("shawon@gmail.com")
    driver.find_element(By.ID, "password").send_keys("12345678")
    driver.find_element(By.ID, "sign-in-btn").click()
    time.sleep(3)
    wait.until(EC.visibility_of_element_located((By.XPATH, "//nav[@aria-label='Staff']")))
    print("Authenticated at:", driver.current_url)

    # Use the app's own logout: sidebar user chip (aria-haspopup=menu) ->
    # "Sign Out" menuitem clears tokens and assigns "/" (verified in AppShell.jsx + StaffApp.jsx).
    wait.until(EC.element_to_be_clickable((By.XPATH, "//button[@aria-haspopup='menu']"))).click()
    time.sleep(1)
    wait.until(EC.element_to_be_clickable((By.XPATH, "//button[@role='menuitem' and contains(., 'Sign Out')]"))).click()
    time.sleep(3)

    # Session must now be invalid: login page shown, tokens cleared
    wait.until(EC.presence_of_element_located((By.ID, "username")))
    token = driver.execute_script("return window.localStorage.getItem('pharvo_access_token');")
    assert token is None, "Access token still present after UI logout."
    print("After logout, login page shown; tokens cleared.")

    # Protected content must now be blocked
    driver.get("http://localhost:5173/pharmacist/dashboard")
    wait.until(EC.presence_of_element_located((By.ID, "username")))
    assert not driver.find_elements(By.XPATH, "//nav[@aria-label='Staff']"), \
        "Protected staff UI accessible after session ended."
    print("Protected route after session end shows: Login.")
    print("PASS: Session handling verified (logout invalidates session; protected page blocked)")
except Exception as e:
    print("FAIL:", e)
    driver.save_screenshot("49_session_expiry_FAIL.png")
finally:
    driver.quit()